# Proyecto 2 - Análisis Exploratorio
## Clasificación Degenerativa de la Columna Lumbar (RSNA 2024)


---
## 1. Carga de los datos


In [18]:
import pandas as pd
import numpy as np

train = pd.read_csv('data/train.csv')
coords = pd.read_csv('data/train_label_coordinates.csv')
series = pd.read_csv('data/train_series_descriptions.csv')

print("train.csv:", train.shape)
print("train_label_coordinates.csv:", coords.shape)
print("train_series_descriptions.csv:", series.shape)

train.csv: (1975, 26)
train_label_coordinates.csv: (48692, 7)
train_series_descriptions.csv: (6294, 3)


## 2. Descripción general de los datos

Antes de limpiar, describimos cuántas variables y observaciones tiene cada dataset, y el tipo de cada variable.

In [35]:
print("train.csv")
print(f"Observaciones (estudios): {train.shape[0]}")
print(f"Variables: {train.shape[1]} (1 identificador + {train.shape[1]-1} etiquetas de severidad)")
print("\nTipos de datos:")
print(train.dtypes.value_counts())
train.info()

train.csv
Observaciones (estudios): 1975
Variables: 26 (1 identificador + 25 etiquetas de severidad)

Tipos de datos:
object    25
int64      1
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1975 entries, 0 to 1974
Data columns (total 26 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   study_id                                1975 non-null   int64 
 1   spinal_canal_stenosis_l1_l2             1974 non-null   object
 2   spinal_canal_stenosis_l2_l3             1974 non-null   object
 3   spinal_canal_stenosis_l3_l4             1974 non-null   object
 4   spinal_canal_stenosis_l4_l5             1974 non-null   object
 5   spinal_canal_stenosis_l5_s1             1974 non-null   object
 6   left_neural_foraminal_narrowing_l1_l2   1973 non-null   object
 7   left_neural_foraminal_narrowing_l2_l3   1973 non-null   object
 8   left_neural_foraminal_narrowing_l3_l4 

In [36]:
print("train_label_coordinates.csv")
coords.info()
print("\ntrain_series_descriptions.csv")
series.info()

train_label_coordinates.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48692 entries, 0 to 48691
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   study_id         48692 non-null  int64  
 1   series_id        48692 non-null  int64  
 2   instance_number  48692 non-null  int64  
 3   condition        48692 non-null  object 
 4   level            48692 non-null  object 
 5   x                48692 non-null  float64
 6   y                48692 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 2.6+ MB

train_series_descriptions.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6294 entries, 0 to 6293
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   study_id            6294 non-null   int64 
 1   series_id           6294 non-null   int64 
 2   series_description  6294 non-null   object
dtypes: int

## 3. Limpieza y preprocesamiento

### 3.1 Verificación de duplicados
Se revisa si hay study_id repetidos en train.csv (no deberían existir, cada estudio es único) y filas exactamente duplicadas en los otros dos archivos.

In [37]:
print("study_id duplicados en train.csv:", train['study_id'].duplicated().sum())
print("Filas duplicadas en train_label_coordinates.csv:", coords.duplicated().sum())
print("Filas duplicadas en train_series_descriptions.csv:", series.duplicated().sum())
print("series_id duplicados en train_series_descriptions.csv:", series['series_id'].duplicated().sum())

study_id duplicados en train.csv: 0
Filas duplicadas en train_label_coordinates.csv: 0
Filas duplicadas en train_series_descriptions.csv: 0
series_id duplicados en train_series_descriptions.csv: 0


**Hallazgo:** No se encontraron duplicados en ninguno de los tres conjuntos de datos.

### 3.2 Validación de tipos y categorías válidas

Se confirma que los valores no nulos de las 25 columnas de severidad sean únicamente uno de los 3 valores esperados (Normal/Mild, Moderate, Severe). Si existiera algún typo o valor inesperado, no se detectaría solo con .info().

In [22]:
valores_esperados = {'Normal/Mild', 'Moderate', 'Severe'}
cols_severidad = [c for c in train.columns if c != 'study_id']

valores_inesperados = {}
for c in cols_severidad:
    valores_col = set(train[c].dropna().unique())
    extra = valores_col - valores_esperados
    if extra:
        valores_inesperados[c] = extra

if valores_inesperados:
    print("Columnas con valores inesperados:", valores_inesperados)
else:
    print("Todas las columnas de severidad usan únicamente los 3 valores esperados: Normal/Mild, Moderate, Severe")

# Verificar que study_id sea numérico consistente en los 3 archivos
print("\nTipo de study_id -> train:", train['study_id'].dtype, "| coords:", coords['study_id'].dtype, "| series:", series['study_id'].dtype)

Todas las columnas de severidad usan únicamente los 3 valores esperados: Normal/Mild, Moderate, Severe

Tipo de study_id -> train: int64 | coords: int64 | series: int64


### 3.3 Valores faltantes en train.csv

Se cuantifican los valores nulos por columna. Esto es clave porque la guía del reto ya advierte que algunas entradas tienen etiquetas incompletas.

In [23]:
nulos = train.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
porcentaje_nulos = (nulos / len(train) * 100).round(2)

resumen_nulos = pd.DataFrame({'nulos': nulos, 'porcentaje': porcentaje_nulos})
print(resumen_nulos)

print(f"\nTotal de filas con al menos un valor faltante: {train.isnull().any(axis=1).sum()} de {len(train)} ({train.isnull().any(axis=1).mean()*100:.1f}%)")

                                        nulos  porcentaje
left_subarticular_stenosis_l1_l2          164        8.30
right_subarticular_stenosis_l1_l2         161        8.15
right_subarticular_stenosis_l2_l3          82        4.15
left_subarticular_stenosis_l2_l3           82        4.15
left_subarticular_stenosis_l5_s1           11        0.56
right_neural_foraminal_narrowing_l3_l4      8        0.41
right_neural_foraminal_narrowing_l1_l2      8        0.41
right_neural_foraminal_narrowing_l5_s1      8        0.41
right_neural_foraminal_narrowing_l4_l5      8        0.41
right_neural_foraminal_narrowing_l2_l3      8        0.41
right_subarticular_stenosis_l5_s1           7        0.35
left_subarticular_stenosis_l3_l4            3        0.15
left_subarticular_stenosis_l4_l5            3        0.15
left_neural_foraminal_narrowing_l5_s1       2        0.10
left_neural_foraminal_narrowing_l4_l5       2        0.10
left_neural_foraminal_narrowing_l3_l4       2        0.10
left_neural_fo

### 3.4 Tabla resumen de nulos por condición y nivel vertebral

Para cumplir con el objetivo específico 1 (completitud de datos por condición y nivel), se transforman las columnas de train.csv a formato largo (melt) y se construye una tabla cruzada de condición × nivel con el porcentaje de nulos en cada combinación.

In [24]:
# Transformación a formato largo: una fila por (study_id, condición, nivel, severidad)
train_long = train.melt(id_vars='study_id', var_name='variable', value_name='severidad')

# Separar 'variable' en condición y nivel (los últimos 2 tokens son el nivel, ej. l1_l2)
def separar_condicion_nivel(nombre_var):
    partes = nombre_var.split('_')
    nivel = '_'.join(partes[-2:])          # ej. l1_l2
    condicion = '_'.join(partes[:-2])      # ej. spinal_canal_stenosis
    return condicion, nivel

train_long[['condicion', 'nivel']] = train_long['variable'].apply(
    lambda v: pd.Series(separar_condicion_nivel(v))
)

train_long = train_long.drop(columns='variable')
print(f"Filas en formato largo: {len(train_long)} (esperado: {len(train)} estudios x 25 variables = {len(train)*25})")
train_long.head()

Filas en formato largo: 49375 (esperado: 1975 estudios x 25 variables = 49375)


,study_id,severidad,condicion,nivel
0,4003253,Normal/Mild,spinal_canal_stenosis,l1_l2
1,4646740,Normal/Mild,spinal_canal_stenosis,l1_l2
2,7143189,Normal/Mild,spinal_canal_stenosis,l1_l2
3,8785691,Normal/Mild,spinal_canal_stenosis,l1_l2
4,10728036,Normal/Mild,spinal_canal_stenosis,l1_l2


In [25]:
# Tabla resumen: porcentaje de nulos por condición y nivel
tabla_nulos = train_long.groupby(['condicion', 'nivel'])['severidad'].apply(
    lambda x: x.isnull().mean() * 100
).unstack().round(2)

tabla_nulos

nivel,l1_l2,l2_l3,l3_l4,l4_l5,l5_s1
condicion,,,,,
left_neural_foraminal_narrowing,0.10,0.10,0.10,0.10,0.10
left_subarticular_stenosis,8.30,4.15,0.15,0.15,0.56
right_neural_foraminal_narrowing,0.41,0.41,0.41,0.41,0.41
right_subarticular_stenosis,8.15,4.15,0.10,0.10,0.35
spinal_canal_stenosis,0.05,0.05,0.05,0.05,0.05




### 3.5 Decisión sobre el manejo de valores faltantes

No se opta por imputar los valores de severidad, ya que se trata de un diagnóstico médico: inventar una severidad introduciría un sesgo clínicamente injustificado. En su lugar:
- Se documentó el porcentaje de datos faltantes por condición y nivel como parte de los hallazgos (tabla anterior).
- Para el análisis de frecuencias y cruces, se excluirán únicamente los registros nulos puntuales de train_long 

In [26]:
# Dataset en formato largo sin nulos, listo para el EDA de frecuencias/cruces
train_long_clean = train_long.dropna(subset=['severidad']).copy()
print(f"Registros originales: {len(train_long)}")
print(f"Registros sin nulos: {len(train_long_clean)} ({len(train_long_clean)/len(train_long)*100:.1f}%)")

Registros originales: 49375
Registros sin nulos: 48803 (98.8%)
